In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu google-generativeai

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import textwrap
import google.generativeai as genai

In [ ]:
dataset = load_dataset("lavita/MedQuAD")
dataset


In [ ]:
print(dataset)

In [ ]:
df = dataset["train"].to_pandas()

In [ ]:
df.columns

In [ ]:
df[["question", "answer", "document_source"]].isna().sum()

In [ ]:
df = df[["question", "answer", "document_source"]].dropna()
df.head()

In [ ]:
len(df)

In [ ]:
documents = []

for idx, row in df.iterrows():
    text = f"""
Question: {row['question']}

Answer: {row['answer']}
"""

    documents.append({
        "id": idx,
        "text": text,
        "source": row["document_source"],
        "question": row["question"]
    })

len(documents)

In [ ]:
print(documents[0]["text"][:1000])

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
small_docs = documents

texts = [doc["text"] for doc in small_docs]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings = embeddings.astype("float32")

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

print("Embeddings shape:", embeddings.shape)

In [ ]:
import pickle

np.save("/kaggle/working/medquad_embeddings_v3.npy", embeddings)

with open("/kaggle/working/medquad_docs_v3.pkl", "wb") as f:
    pickle.dump(small_docs, f)

print("Embeddings and documents saved successfully.")

In [ ]:
dimension = embeddings.shape[1]

# IndexFlatIP + normalized embeddings = cosine similarity
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

In [ ]:
!pip install langchain_groq

In [ ]:
import os
from langchain_groq import ChatGroq

In [ ]:
os.environ["GROQ_API_KEY"] = "enter your api key here"


In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

In [ ]:
def expand_query(query):
    prompt = f"""
You are a medical query expansion assistant for a RAG retriever.

Your job is to convert the user's question into a better search query.

Rules:
- Do not answer the question.
- Keep the original meaning.
- Keep common patient words and add medical synonyms.
- Prefer short keyword-style phrases, not a full sentence.
- Use both patient-friendly words and medical terms.
- Add only highly relevant medical terms.
- Do not add too many possible diseases.
- If the question is not medical, return the original question unchanged.
- Return only one line.

User Question:
{query}

Expanded Search Query:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
def retrieve_docs(query, top_k=4, fetch_k=10, use_query_expansion=True):
    if use_query_expansion:
        search_query = expand_query(query)
    else:
        search_query = query

    query_embedding = embedding_model.encode([search_query], convert_to_numpy=True)
    query_embedding = query_embedding.astype("float32")

    # Normalize query embedding for cosine similarity
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, fetch_k)

    results = []
    seen_questions = set()

    for score, idx in zip(scores[0], indices[0]):
        doc = small_docs[idx]

        question_key = doc["question"].strip().lower()

        if question_key in seen_questions:
            continue

        seen_questions.add(question_key)

        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "score": float(score),
            "expanded_query": search_query
        })

        if len(results) == top_k:
            break

    return results

In [ ]:
def is_relevant(retrieved_docs, threshold=0.35):
    if len(retrieved_docs) == 0:
        return False

    best_score = retrieved_docs[0]["score"]

    return best_score >= threshold

In [ ]:
def filter_relevant_sources(retrieved_docs, min_score=0.30):
    filtered_docs = []

    for doc in retrieved_docs:
        if doc["score"] >= min_score:
            filtered_docs.append(doc)

    return filtered_docs

In [ ]:
def assess_retrieval_quality(retrieved_docs, threshold=0.40, min_good_sources=1):
    """
    Checks if retrieved documents are good enough for answer generation.
    Since we are using cosine similarity, higher score means better match.
    """
    
    if len(retrieved_docs) == 0:
        return {
            "is_good": False,
            "reason": "No documents were retrieved.",
            "good_sources_count": 0,
            "best_score": 0.0
        }
    
    good_sources = [
        doc for doc in retrieved_docs
        if doc["score"] >= threshold
    ]
    
    best_score = retrieved_docs[0]["score"]
    
    if len(good_sources) >= min_good_sources:
        return {
            "is_good": True,
            "reason": "Retrieval quality is good enough.",
            "good_sources_count": len(good_sources),
            "best_score": best_score
        }
    
    return {
        "is_good": False,
        "reason": f"Only {len(good_sources)} sources passed the threshold.",
        "good_sources_count": len(good_sources),
        "best_score": best_score
    }

In [ ]:
def rewrite_query_for_correction(query, previous_expanded_query=None):
    """
    Rewrites the query only when retrieval quality is weak.
    """
    
    prompt = f"""
You are a medical query rewriting assistant for a corrective RAG system.

The previous retrieval was weak. Rewrite the user's question into a better medical search query.

Rules:
- Do not answer the question.
- Keep the original meaning.
- Keep common patient words and add medical synonyms.
- Use simple keyword-style phrases.
- Include important symptoms, disease names, body parts, and medical terms.
- Do not add rare diseases unless clearly mentioned by the user.
- If the question is not medical, return the original question unchanged.
- Return only one line.

Original User Question:
{query}

Previous Expanded Query:
{previous_expanded_query}

Corrected Search Query:
"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [ ]:
def search_vector_db(search_query, top_k=3, fetch_k=10):
    """
    Searches FAISS using a provided query string.
    """
    
    query_embedding = embedding_model.encode([search_query], convert_to_numpy=True)
    query_embedding = query_embedding.astype("float32")
    
    faiss.normalize_L2(query_embedding)
    
    scores, indices = index.search(query_embedding, fetch_k)
    
    results = []
    seen_questions = set()
    
    for score, idx in zip(scores[0], indices[0]):
        doc = small_docs[idx]
        
        question_key = doc["question"].strip().lower()
        
        if question_key in seen_questions:
            continue
        
        seen_questions.add(question_key)
        
        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "score": float(score),
            "searched_query": search_query
        })
        
        if len(results) == top_k:
            break
    
    return results

In [ ]:
def generate_answer(query, retrieved_docs):
    context = "\n\n".join([
        f"[Source {i+1}]\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""
You are a medical document assistant.

Answer the user's question using ONLY the provided sources.

Rules:
1. Use only the information from the provided sources.
2. Add citations in the answer using [Source 1], [Source 2], etc.
3. If the sources do not contain enough information, say:
   "The provided medical documents do not contain enough information."
4. Keep the answer simple and clear.
5. Do not diagnose the user.
6. Suggest consulting a healthcare professional when appropriate.

Sources:
{context}

User question:
{query}

Answer with citations:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
def corrective_retrieve(
    query,
    top_k=3,
    fetch_k=10,
    threshold=0.40,
    max_corrections=1,
    use_query_expansion=True
):
    """
    Version 4 corrective retrieval:
    1. Retrieve using V3 retriever.
    2. Assess retrieval quality.
    3. If weak, rewrite query and retrieve again.
    """
    
    correction_count = 0
    
    # First retrieval using V3 function
    retrieved_docs = retrieve_docs(
        query=query,
        top_k=top_k,
        fetch_k=fetch_k,
        use_query_expansion=use_query_expansion
    )
    
    expanded_query = retrieved_docs[0].get("expanded_query", query) if len(retrieved_docs) > 0 else query
    
    quality = assess_retrieval_quality(
        retrieved_docs=retrieved_docs,
        threshold=threshold,
        min_good_sources=1
    )
    
    # If weak, correct query and retrieve again
    while not quality["is_good"] and correction_count < max_corrections:
        correction_count += 1
        
        corrected_query = rewrite_query_for_correction(
            query=query,
            previous_expanded_query=expanded_query
        )
        
        corrected_docs = search_vector_db(
            search_query=corrected_query,
            top_k=top_k,
            fetch_k=fetch_k
        )
        
        for doc in corrected_docs:
            doc["expanded_query"] = expanded_query
            doc["corrected_query"] = corrected_query
        
        retrieved_docs = corrected_docs
        
        quality = assess_retrieval_quality(
            retrieved_docs=retrieved_docs,
            threshold=threshold,
            min_good_sources=1
        )
    
    return {
        "retrieved_docs": retrieved_docs,
        "quality": quality,
        "correction_count": correction_count
    }

In [ ]:
def rag_chatbot(
    query,
    top_k=3,
    threshold=0.40,
    min_source_score=0.50,
    use_query_expansion=True,
    max_corrections=1
):
    retrieval_output = corrective_retrieve(
        query=query,
        top_k=top_k,
        fetch_k=10,
        threshold=threshold,
        max_corrections=max_corrections,
        use_query_expansion=use_query_expansion
    )
    
    retrieved_docs = retrieval_output["retrieved_docs"]
    quality = retrieval_output["quality"]
    correction_count = retrieval_output["correction_count"]
    
    print("Question:")
    print(query)
    
    if len(retrieved_docs) > 0:
        print("\nExpanded Query:")
        print(retrieved_docs[0].get("expanded_query", query))
        
        if correction_count > 0:
            print("\nCorrected Query:")
            print(retrieved_docs[0].get("corrected_query", "No corrected query available."))
    
    print("\nRetrieval Quality:")
    print("Best Score:", quality["best_score"])
    print("Good Sources Count:", quality["good_sources_count"])
    print("Correction Count:", correction_count)
    print("Quality Reason:", quality["reason"])
    
    if not quality["is_good"]:
        print("\nAnswer:")
        print("The provided medical documents do not contain enough information.")
        return
    
    filtered_docs = filter_relevant_sources(
        retrieved_docs,
        min_score=min_source_score
    )
    
    if len(filtered_docs) == 0:
        print("\nAnswer:")
        print("The provided medical documents do not contain enough information.")
        print("\nReason:")
        print("Retrieved documents were below the minimum source score.")
        return
    
    answer = generate_answer(query, filtered_docs)
    
    print("\nAnswer:")
    print(answer)
    
    print("\nRetrieved Sources:")
    for i, doc in enumerate(filtered_docs, 1):
        print(f"\n--- Source {i} ---")
        print("Dataset Source:", doc["source"])
        print("Matched Question:", doc["question"])
        print("Similarity Score:", doc["score"])
        print("Searched Query:", doc.get("searched_query", "N/A"))

In [ ]:
test_questions = [
    # Familiar / easy medical questions
    "What are the symptoms of diabetes?",
    "How is high blood pressure treated?",
    "What is glaucoma?",

    # Patient-style unseen questions
    "What are signs of sugar disease?",
    "I feel strange after taking diabetes medicine but I did not eat.",
    "My pressure is high but I do not feel anything. Is that dangerous?",
    "My eyes are slowly getting weak and side vision is decreasing.",
    "Baby has diarrhea and dry mouth and no wet diaper.",

    # Biomedical-style questions
    "What mechanisms explain vision impairment in diabetic eye disease?",
    "How is hypoglycemia related to diabetes medication and missed meals?",
    "How does hypertension contribute to kidney disease?",

    # Weak/vague medical questions
    "My sugar is acting weird and I feel strange.",
    "Blood pressure medicine and kidney protection.",
    "I feel weak, sweaty, and confused after skipping food.",
    "Eye pressure, tunnel vision, and vision loss.",

    # Out-of-scope questions
    "Who won the FIFA World Cup?",
    "How do I train a machine learning model?",
    "Write Python code for bubble sort.",
    "What is the capital of Germany?"
]

for q in test_questions:
    print("="*100)
    rag_chatbot(q)

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/medquad_embeddings_v3.npy")